# 19 — Export report inputs

Copy only persisted evidence into a checksum-protected report bundle. Report-writing prompts should read this directory rather than notebook prose.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.reporting import build_report_bundle


In [ ]:
files = [
    PATHS.processed / "fuel_annual_analytical_panel.csv",
    PATHS.processed / "fuel_monthly_analytical_panel.csv",
    PATHS.metrics / "descriptive_metrics.csv",
    PATHS.metrics / "dependence_pre_post_summary.csv",
    PATHS.metrics / "structural_break_tests.csv",
    PATHS.metrics / "annual_interrupted_trend_models.csv",
    PATHS.metrics / "exploratory_break_sup_wald.csv",
    PATHS.metrics / "annual_model_residual_diagnostics.csv",
    PATHS.metrics / "stress_2022_metrics.csv",
    PATHS.metrics / "stress_2022_source_sensitivity.csv",
    PATHS.metrics / "annual_source_sensitivity.csv",
    PATHS.metrics / "monthly_event_phase_summary.csv",
    PATHS.metrics / "monthly_event_models.csv",
    PATHS.metrics / "event_window_sensitivity.csv",
    # These gate publication in notebook 20, so the report writer must be able
    # to see the evidence that unblocked the report.
    PATHS.metrics / "jodi_trade_annual_completeness.csv",
    PATHS.metrics / "jodi_demand_annual_completeness.csv",
    PATHS.metrics / "jodi_refinery_output_annual_completeness.csv",
    PATHS.tables / "headline_event_years.csv",
    PATHS.reference / "refinery_events.csv",
    PATHS.provenance / "source_manifest_snapshot.csv",
    PATHS.provenance / "software_environment.csv",
]
# Price files are optional until the EC workbook has been transformed.
for optional in [
    PATHS.metrics / "price_comovement_models.csv",
    PATHS.metrics / "price_stationarity_diagnostics.csv",
    PATHS.metrics / "price_model_choice.csv",
    PATHS.metrics / "price_adf_lag_sensitivity.csv",
    PATHS.metrics / "price_model_choice_scale_comparison.csv",
    PATHS.metrics / "price_short_run_models.csv",
    PATHS.metrics / "price_ecm_models.csv",
    PATHS.metrics / "price_ecm_half_lives.csv",
    PATHS.metrics / "pt_es_spread_stationarity.csv",
    PATHS.metrics / "pt_es_spread_stationarity_by_regime.csv",
    PATHS.metrics / "price_kpss_diagnostics.csv",
    PATHS.metrics / "weekly_price_coverage.csv",
    PATHS.metrics / "price_cointegrating_slope_tests.csv",
    PATHS.metrics / "price_elasticity_unit_tests.csv",
    PATHS.metrics / "price_post_period_stability.csv",
    PATHS.metrics / "pt_es_price_spread_summary.csv",
    PATHS.metrics / "pt_es_physical_balance_comparison.csv",
    PATHS.metrics / "jodi_dgeg_trade_reconciliation.csv",
    PATHS.metrics / "jodi_eurostat_trade_reconciliation.csv",
    PATHS.metrics / "jodi_eurostat_reconciliation_summary.csv",
    PATHS.metrics / "monthly_annual_balance_reconciliation.csv",
    PATHS.metrics / "monthly_annual_agreement_summary.csv",
    PATHS.processed / "eurostat_physical_balance_panel.csv",
]:
    if optional.exists():
        files.append(optional)
# Figures go through the manifest too. Copying them afterwards left files in a
# checksum-protected directory that the manifest did not describe.
figures = sorted(PATHS.figures.glob("*.png"))
manifest = build_report_bundle(
    destination=PATHS.report_inputs,
    files=files,
    figures=figures,
    required=True,
)
manifest


In [ ]:
print(f"Report bundle: {PATHS.report_inputs}")
print(f"  {len(manifest['files'])} manifested files, {len(manifest['missing'])} missing")
if manifest["removed_unmanifested"]:
    print(f"  removed {len(manifest['removed_unmanifested'])} stale unmanifested file(s)")